# 01 · Data Overview

**Amaç:** Veri sözleşmesini doğrulamak, temel istatistikleri çıkarmak. Bu notebook `scripts/00_inspect_data.py` ile aynı bilgileri görsel olarak doğrular.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

from src.data.loader import load_validated


In [ ]:
df, schema = load_validated()
print('Schema OK:', schema.ok)
print('Shape:', df.shape)
df.head(3)

## Dtypes & Cardinality

In [ ]:
dtypes = df.dtypes.value_counts()
print('Dtype kompozisyonu:')
print(dtypes)

card = df.nunique().sort_values(ascending=False)
card.to_frame('unique').head(15)

## Eksik değerler

In [ ]:
m = df.isna().sum()
m = m[m>0].sort_values(ascending=False)
m.to_frame('n').assign(pct=lambda x: (x['n']/len(df)*100).round(2))

## Target dağılımı

In [ ]:
tgt = df['IsFraudTransaction'].value_counts()
print(tgt)
print(f'Fraud rate: {df["IsFraudTransaction"].mean()*100:.4f}%')
fig, ax = plt.subplots()
tgt.plot.bar(ax=ax)
ax.set_title('Target distribution (note severe imbalance)')
ax.set_yscale('log')

## Tarih aralığı

In [ ]:
print('Min:', df['TransactionDate'].min())
print('Max:', df['TransactionDate'].max())
print('Span:', (df['TransactionDate'].max()-df['TransactionDate'].min()).days, 'days')

## TransactionChannel — atılacak (tek değerli)

In [ ]:
# Loader bunu zaten drop'luyor (configs/data.yaml). Burada doğrulama:
print('TransactionChannel kolonu df içinde mi?', 'TransactionChannel' in df.columns)

## Sonuç
- Şema birebir uyumlu. 849.564 satır × 25 kolon (TransactionChannel drop'lanmış).
- Fraud rate ~%0.63 → ciddi imbalance.
- DayType %96 NaN — sonraki notebook'larda iki varyantta test edeceğiz.
- Tarih aralığı 727 gün.